# Libreria Geoespacial

La libreria que utulizaremos para hacer nuestro mapa interactivo es **folium**. El proposito de este archivo sera crear un mapa interactivo que nos permita visualizar las estaciones de eco bicis por medio de una vista geoespacial.

In [1]:
#pip install folium

In [2]:
import folium
import os
import gc
import pandas as pd

## Numero de Viajes

Trabajaremos con el archivo parquet que habiamos hecho en el codigo anterior.

Un archivo **Parquet** es un formato de almacenamiento de datos columnar, de código abierto y altamente eficiente, diseñado para gestionar grandes volúmenes de información 

In [3]:
df = pd.read_parquet('df_final_G28.parquet')

In [4]:
# Vamos a contar las frecuencias de cada estacion, 
# primero para el retiro y despues para el arribo
ret=df['Ciclo_Estacion_Retiro'].value_counts()
arr=df['Ciclo_Estacion_Arribo'].value_counts()

In [5]:
# Verificamos que no contenga valores NaN
ret.isna().sum()
#arr.isna().sum()

np.int64(0)

In [6]:
#Vamos a transformarlos en un dataframe
d_ret=ret.reset_index()

In [7]:
# Se renombran las columnas para evitar confusiones
d_ret.columns = ['Ciclo_Estacion', 'Frecuencia_Total']
# Lo acomodamos de mayor a menor
d_ret = d_ret.sort_values(by='Frecuencia_Total', ascending=False)

Esto es lo que vamos a colocar dentro de la estacion de ecobicis, vamos a empalmar Ciclo Estacion con el ID y colocar su respectiva frecuencia

## Mapa

In [8]:
os.getcwd()

'/home/marco-carrillo/Descargas/Repaso/Repositorio'

In [9]:
os.listdir()

['estaciones-de-ecobici.csv',
 'df_unid_G28.parquet',
 '.ipynb_checkpoints',
 'Mapa Geoespacial Ecobicis.ipynb',
 'df_final_G28.parquet']

In [10]:
base= pd.read_csv('estaciones-de-ecobici.csv')

In [11]:
base.head(3)

,ID,Nombre,Dirección,Número,Código postal,districtCode,Colonia,altitude,nearbyStations/0,location/lat,location/lon,Tipo de estación,nearbyStations/1,nearbyStations/2,nearbyStations/3,nearbyStations/4,nearbyStations/5,punto_geo
0,416,416 RECREO-PARROQUÍA,416 - Recreo-Parroquía,S/N,3104.0,1,Ampliación Granada,NaN,390,19.371308,-99.177607,BIKE,391.0,415.0,NaN,NaN,NaN,"19.371308,-99.177607"
1,440,440 REAL DE MAYORAZGO-PUENTE XOCO,440 - Real De Mayorazgo-Puente Xoco,S/N,3330.0,1,Ampliación Granada,NaN,433,19.360650,-99.168669,BIKE,439.0,441.0,NaN,NaN,NaN,"19.36065,-99.168669"
2,70,70 PARQUE MEXICO-MICHOACAN,070 - Parque México - Michoacan,S/N,6100.0,1,Ampliación Granada,NaN,69,19.411272,-99.169718,"BIKE,TPV",71.0,76.0,NaN,NaN,NaN,"19.411272,-99.169718"


Vamos a realizar un merge de la base que cargamos anteriormente con el numero de viajes, con esta nueva base que contiene la localizacion de cada estacion

In [12]:
# Emparejamos los valores de Ciclo_Estacion de df_tot_fq y ID del df de estaciones
base = base.merge(d_ret, left_on='ID', right_on='Ciclo_Estacion', how='left' )

In [13]:
base.head(3)

,ID,Nombre,Dirección,Número,Código postal,districtCode,Colonia,altitude,nearbyStations/0,location/lat,location/lon,Tipo de estación,nearbyStations/1,nearbyStations/2,nearbyStations/3,nearbyStations/4,nearbyStations/5,punto_geo,Ciclo_Estacion,Frecuencia_Total
0,416,416 RECREO-PARROQUÍA,416 - Recreo-Parroquía,S/N,3104.0,1,Ampliación Granada,NaN,390,19.371308,-99.177607,BIKE,391.0,415.0,NaN,NaN,NaN,"19.371308,-99.177607",NaN,NaN
1,440,440 REAL DE MAYORAZGO-PUENTE XOCO,440 - Real De Mayorazgo-Puente Xoco,S/N,3330.0,1,Ampliación Granada,NaN,433,19.360650,-99.168669,BIKE,439.0,441.0,NaN,NaN,NaN,"19.36065,-99.168669",NaN,NaN
2,70,70 PARQUE MEXICO-MICHOACAN,070 - Parque México - Michoacan,S/N,6100.0,1,Ampliación Granada,NaN,69,19.411272,-99.169718,"BIKE,TPV",71.0,76.0,NaN,NaN,NaN,"19.411272,-99.169718",70.0,52016.0


Vamos a corroborar cuantas estaciones tienen asociada una frecuencia y cuantas no

In [14]:
print("Total de estaciones:", len(base))
print("Con frecuencia:", base['Frecuencia_Total'].notna().sum())
print("Sin frecuencia:", base['Frecuencia_Total'].isna().sum())

Total de estaciones: 480
Con frecuencia: 273
Sin frecuencia: 207


Sabemos que son un total de 480 estaciones registradas, para colocarlas dentro del mapa necesitamos las columnas de latitud y longitud

Primero tenemos que definir el mapa que vamos a trabajar, para nuesto caso lo que haremos sera aproximarnos a las coordenas del **Angel de la Independencia**

In [15]:
Angel = folium.Map(location=[19.4326, -99.1625], zoom_start=18)

## Tipo de Mapa

A continuacion se presentaran diferentes formatosde mapa que buscan cumplit el obejtivo de este codigo

### Mapa de Despliegue Sencillo

In [16]:
#Creamos una lista par guardar los marcadores que iremos creando para cada estación.
pos = []

#Definimos el loop donde se va a colocar el nombre de cada estacion
for _, fila in base.iterrows():

    #Extraemos las coordenadas
    lat = fila['location/lat']
    lon = fila['location/lon']
    nombre = fila['Nombre']
    frecuencia = fila['Frecuencia_Total']

    #Esto es para aquellas estaciones de las que no tenemos datos
    if pd.isna(frecuencia):
        texto_frecuencia = "Sin datos"
    else:
        texto_frecuencia = f"{int(frecuencia):,}"

    #Con esto creamos el marcador

    eco = folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(
            f"""
            <b>Estación:</b> {nombre}<br>
            <b>Número de retiros:</b> {texto_frecuencia}
            """,
            max_width=300
        ),
        tooltip=nombre
    )

    #se agregan los marcadores al Mapa
    eco.add_to(Angel)
    pos.append(eco)

Lo unico que tenemos que hacer ahora es ejecutar el mapa y darle click a una de las estaciones para que nos despliegue la información

In [17]:
Angel

### Mapa de Despliegue con colores segun la demanda

In [18]:
# Calculamos los límites de demanda
q1 = base['Frecuencia_Total'].quantile(1/3)
q2 = base['Frecuencia_Total'].quantile(2/3)

In [19]:
# Mapa nuevo
Angel2 = folium.Map(location=[19.40, -99.16],zoom_start=12)

In [20]:
#Lista
pos = []
#Proceso previo pero agregando los colores
for _, fila in base.iterrows():

    lat = fila['location/lat']
    lon = fila['location/lon']
    nombre = fila['Nombre']
    frecuencia = fila['Frecuencia_Total']

    if pd.isna(frecuencia):
        color = 'gray'
        texto = 'Sin datos'

    elif frecuencia <= q1:
        color = 'green'
        texto = f'{int(frecuencia):,}'

    elif frecuencia <= q2:
        color = 'orange'
        texto = f'{int(frecuencia):,}'

    else:
        color = 'red'
        texto = f'{int(frecuencia):,}'

    eco = folium.CircleMarker(
        location=[lat, lon],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.75,
        popup=folium.Popup(
            f'<b>Estación:</b> {nombre}<br>'
            f'<b>Retiros:</b> {texto}',
            max_width=300
        ),
        tooltip=nombre
    )

    eco.add_to(Angel2)
    pos.append(eco)


In [21]:
Angel2

### Raking de Estaciones más utilizadas

Primero vamos a crear un Rankign con los 10 más usados, esto lo incorporaremso al mapa despues

In [22]:
top10 = ( base[['Nombre', 'Frecuencia_Total']].dropna(subset=['Frecuencia_Total'])
    .sort_values('Frecuencia_Total', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

Ahora avamos a incorporar las leyendas al mapa para que sea más entendible que representa cada una

In [23]:
from branca.element import Element

leyenda = """
<div style="
    position: fixed;
    bottom: 25px;
    left: 15px;
    width: 190px;
    background-color: white;
    border: 2px solid #555;
    border-radius: 8px;
    padding: 12px;
    z-index: 9999;
    font-size: 13px;
">
    <h4 style="margin: 0 0 10px 0;">
        Demanda de Ecobici
    </h4>

    <div style="margin-bottom: 7px;">
        <span style="color: green; font-size: 20px;">●</span>
        Demanda baja
    </div>

    <div style="margin-bottom: 7px;">
        <span style="color: orange; font-size: 20px;">●</span>
        Demanda media
    </div>

    <div style="margin-bottom: 7px;">
        <span style="color: red; font-size: 20px;">●</span>
        Demanda alta
    </div>

    <div>
        <span style="color: gray; font-size: 20px;">●</span>
        Sin datos
    </div>

    <hr style="margin: 10px 0;">

    <small>
        Clasificación relativa según la frecuencia
        de retiros registrada.
    </small>
</div>
"""

Angel2.get_root().html.add_child(Element(leyenda))

Y en este ultimo para solo agregaremos el Raking que anteriormente ya habiamos definido

In [24]:
from branca.element import Element

filas_html = ""

for i, fila in top10.iterrows():
    filas_html += f"""
    <tr>
        <td style="padding:4px 8px;">{i + 1}</td>
        <td style="padding:4px 8px;">{fila['Nombre']}</td>
        <td style="padding:4px 8px; text-align:right;">
            {int(fila['Frecuencia_Total']):,}
        </td>
    </tr>
    """

panel = f"""
<div style="
    position: fixed;
    top: 15px;
    right: 15px;
    width: 380px;
    max-height: 65vh;
    overflow-y: auto;
    background-color: white;
    border: 2px solid #555;
    border-radius: 8px;
    padding: 12px;
    z-index: 9999;
    font-size: 12px;
">
    <h4 style="margin-top:0;">
        Top 10 estaciones por retiros
    </h4>
    <table style="width:100%; border-collapse:collapse;">
        <tr style="border-bottom:2px solid #ddd;">
            <th>#</th>
            <th>Estación</th>
            <th>Retiros</th>
        </tr>
        {filas_html}
    </table>
</div>
"""

Angel2.get_root().html.add_child(Element(panel))

In [25]:
Angel2